# Loading and exploring the RIP Toolkit

This toolkit uses RIP through a container, which saves you the trouble of installing it on your system.
To be able to use it, you need to have access to `apptainer`, and the container image available on your system.

In [ ]:
import wrf_analysis_toolkit as wat
import rip_toolkit as ript
import os
os.environ["LOAD_APPTAINER_MODULE"] = "1"

image_path = "/mnt/projects/wrf-storm-workshop/ripdocker_latest.sif"

In [ ]:
print(ript.__version__)

To get all the functions defined in the module, call:

In [ ]:
print(ript.__all__)

And you can get the documentation for that function by running:

In [ ]:
help(ript.preprocess)

## The usual RIP workflow

The RIP toolkit simplifies the workflow for generating trajectory plots from WRF output files into three steps:
1. Preprocess the WRF output files to generate the necessary input files for RIP (RIPDP).
   - This step is done by the `preprocess` function, and only needs to be done once for a given set of WRF output files.
2. Locate the points of interest using a terrain plot (see WRF Analysis Toolkit).
3. Identify the model times of interest for the trajectory analysis.
   - The relevant function in the toolkit is `get_model_times`, which maps model times to date_time.
4. Generate the trajectory data using RIP.
   - This uses the ripdp data generated in the previous step, and can be done multiple times for different sets of trajectories.
   - The relevant functions in the toolkit are `point_trajectory` and `swarm_trajectories`.
   - Optionally, diagnostics can be calculated along the trajectories. These are saved in a csv file.
5. Generate the a plot using the trajectory.
   - This uses the trajectory data generated in the previous step, and can combine multiple trajectories into a single plot.

# Preprocessing WRF output files into RIPDP data

First, lets define where our data comes from, and where we will save outputs.

In [ ]:
wrfout_dir="/mnt/projects/wrf-storm-workshop/data/wrfout/sample"
output_dir = "rip_results/sample"
file_tag = "sample"

To preprocess the whole dataset, we call the `preprocess` function with:

In [ ]:
ripdp = ript.preprocess(
    wrfout_dir=wrfout_dir,
    output_dir=output_dir,
    file_tag=file_tag,
    image_path=image_path,
)
print(ripdp)

The ouptput is the path to the ripdp input file, which will be used in the next step to generate trajectories.

For real datasets, this tends to be very time consuming, so for this workshop we have already run this step for the datasets in `/mnt/projects/wrf-storm-workshop/data/wrfout/`.

You will find the output ripdp files in `/mnt/projects/wrf-storm-workshop/data/ripdp/`.

**Note:** Make sure you point to *the file inside* the `ripdp` directory, and not only to the directory.

# Locating points of interest

RIP only works with grid coordinates, so we need to look at a terrain plot with the `region_ticks` parameter set to `True`.

In [ ]:
f = wat.terrain(
    wrfout_dir=wrfout_dir,
    output_dir=output_dir,
    region_ticks=True,
)

print(f"{output_dir}/{f}.pdf")

Grid ticks are shown on the bottom and right axes of the plot, and can be used to locate the "release" (i.e., start) point of the trajectories.

# Identifying model times of interest

Model times are measured in hours after the start of the simulation, and can be obtained from the `get_model_times` function:

In [ ]:
mt=ript.get_model_times(wrfout_dir)
ript.print_model_times(mt)

All the inputs and ouputs of RIP are in model time, so it is important to know the model times of interest for your analysis.

You may want to use the reversed dictionary to map date strings to lists of model hours safely. This can be done using the `date_model_times` function:

In [ ]:
dmt = ript.date_model_times(mt)
dmt["2005-08-28_03:00:00"]

# Single Point Trajectories

To generate a trajectory from a single point, we call the `point_trajectory` function with the `x`,`y` grid coordinates of the starting point, its *elevation* (in hPa), the start and end model times of the trajectory, and the size of the time step used in the computation (in seconds).

Additionally, we can specify which diagnostics to calculate along the trajectory. There are already a few groups of diagnostics, which can be obtained with the `diagnostic_groups` function, but you may define your own.

In [ ]:
pt_y = ript.point_trajectory(
    wrfout_dir=wrfout_dir,
    output_dir=output_dir,
    ripdp_data=ripdp,
    traj_tag="yucatan_900_hPa",
    traj_x=48,
    traj_y=17,
    traj_z=900,
    traj_t_0=0,
    traj_t_f=12,
    traj_dt=1200,
    hydrometeor=0,
    traj_diagnostics=ript.diagnostic_groups("base"),
    image_path=image_path,
)
print(pt_y)

The output of this function is the trajectory (`.traj`) file. The `csv` file with the diagnostics is generated in the same directory as the trajectory file, with only different file extension.

**Note:** The times (`t_0`, `t_f`) need to exactly match the model times, make sure you use the same decimal places too!

Trajectories can have a `t_f` which is later than `t_0`. In that case, the trajectory "release" point is actually the end point of the trajectory, and the trajectory is calculated backwards in time.

In [ ]:
pt_f = ript.point_trajectory(
    wrfout_dir=wrfout_dir,
    output_dir=output_dir,
    ripdp_data=ripdp,
    traj_tag="florida_900_hPa",
    traj_x=80,
    traj_y=40,
    traj_z=900,
    traj_t_0=12,
    traj_t_f=0,
    traj_dt=1200,
    hydrometeor=0,
    traj_diagnostics=ript.diagnostic_groups("base"),
    image_path=image_path,
)
print(pt_f)

# Generating plots from trajectories

This function assumes you have already generated the trajectory files, and will try and get the information (and file paths) simply from the `trajectory_tag` you used to generate the trajectory files.

To generate the plot, it will take a dictionary, with the trajectory tags as keys, and the colors to use for each trajectory as values.

In [ ]:
pl = ript.plot_trajectories(
    output_dir=output_dir,
    ripdp_data=ripdp,
    traj_tags_colors={"yucatan_900_hPa": "green", "florida_900_hPa": "blue"},
    plot_tag="Sample_plot",
    image_path=image_path,
    format="pdf",
)
print(pl)

The output of this function is the path to the generated plot, which will be saved in the specified `output_dir`.

# Swarm trajectories

In many cases you will want to actually track a column of air, or see how a grid of points evolves over time.
The `swarm_trajectories` function is provided to do this.
It works in almost the same way as the point trajectory function, but instead of single x,y,z values, it takes arrays of x, y, and z values.

The function will generate a trajectory for each combination of x, y, and z values, and return a dictionary of trajectory tags and colors.
This dictionary can then be passed to the `plot_trajectories` function to generate a plot of the stacked trajectories.

In [ ]:
sw = ript.swarm_trajectories(
    wrfout_dir=wrfout_dir,
    output_dir=output_dir,
    ripdp_data=ripdp,
    traj_tag="gulf_of_mexico",
    traj_x=[50, 65],
    traj_y=[30, 45],
    traj_z=[900, 600],
    traj_t_0=0,
    traj_t_f=6,
    traj_dt=1200,
    hydrometeor=0,
    traj_diagnostics=ript.diagnostic_groups("base"),
    image_path=image_path,
)

In [ ]:
pl = ript.plot_trajectories(
    output_dir=output_dir,
    ripdp_data=ripdp,
    traj_tags_colors=sw,
    plot_tag="Sample_swarm_plot",
    image_path=image_path,
    format="pdf",
)
print(pl)


Since tracking the column of air is the most common use case, preset elevations are provided; if not passed, the function then produces trajectories for the stack:


In [ ]:
st = ript.swarm_trajectories(
    wrfout_dir=wrfout_dir,
    output_dir=output_dir,
    ripdp_data=ripdp,
    traj_tag="gulf_of_mexico",
    traj_x=[60],
    traj_y=[30],
    traj_t_0=0,
    traj_t_f=6,
    traj_dt=1200,
    hydrometeor=0,
    traj_diagnostics=ript.diagnostic_groups("base"),
    image_path=image_path,
)

In [ ]:
pl = ript.plot_trajectories(
    output_dir=output_dir,
    ripdp_data=ripdp,
    traj_tags_colors=st,
    plot_tag="Sample_stack_plot",
    image_path=image_path,
    format="pdf",
)
print(pl)